# Izgradnja s Meta obiteljskim modelima 

## Uvod 

Ova lekcija će obuhvatiti: 

- Istraživanje dva glavna Meta obiteljska modela - Llama 3.1 i Llama 3.2 
- Razumijevanje scenarija i slučajeva korištenja za svaki model 
- Primjer koda za prikaz jedinstvenih značajki svakog modela 


## Meta obitelj modela 

U ovoj lekciji istražit ćemo 2 modela iz Meta obitelji ili "Llama jata" - Llama 3.1 i Llama 3.2 

Ovi modeli dolaze u različitim varijantama i dostupni su u [Microsoft Foundry Models katalogu](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Napomena:** GitHub modeli se ukidaju krajem srpnja 2026. Više detalja o korištenju [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) za prototipiziranje s AI modelima potražite ovdje.

Varijante modela: 
- Llama 3.1 - 70B Instruct 
- Llama 3.1 - 405B Instruct 
- Llama 3.2 - 11B Vision Instruct 
- Llama 3.2 - 90B Vision Instruct 

*Napomena: Llama 3 je također dostupna u Microsoft Foundry Models, ali neće biti obrađena u ovoj lekciji*



## Llama 3.1 

S 405 milijardi parametara, Llama 3.1 spada u kategoriju otvorenih LLM-ova. 

Ova je verzija nadogradnja ranijeg izdanja Llama 3, nudeći: 

- Veće kontekstualno područje - 128k tokena naspram 8k tokena 
- Veći maksimalni broj izlaznih tokena - 4096 naspram 2048 
- Bolju podršku za više jezika - zbog povećanja broja trening tokena 

Ovo omogućuje Llama 3.1 da rukuje složenijim slučajevima korištenja prilikom izrade GenAI aplikacija, uključujući: 
- Izvorno pozivanje funkcija - sposobnost pozivanja vanjskih alata i funkcija izvan LLM radnog toka
- Bolje RAG performanse - zahvaljujući većem kontekstualnom području 
- Generiranje sintetičkih podataka - sposobnost kreiranja učinkovitih podataka za zadatke poput finog podešavanja 



### Izvorno pozivanje funkcija 

Llama 3.1 je dodatno podešen da učinkovitije poziva funkcije ili alate. Također ima dva ugrađena alata koje model može prepoznati kao potrebne za korištenje na temelju korisničkog upita. Ti alati su: 

- **Brave Search** - Može se koristiti za dobivanje ažurnih informacija poput vremenske prognoze putem web pretraživanja 
- **Wolfram Alpha** - Može se koristiti za složenije matematičke izračune, pa nije potrebno pisati vlastite funkcije. 

Također možete izraditi vlastite prilagođene alate koje LLM može pozivati. 

U sljedećem primjeru koda: 

- Definiramo dostupne alate (brave_search, wolfram_alpha) u sustavnom promptu. 
- Šaljemo korisnički upit koji pita o vremenu u određenom gradu. 
- LLM će odgovoriti pozivom alata Brave Search koji će izgledati ovako `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Napomena: Ovaj primjer samo poziva alat, ako želite dobiti rezultate, trebate napraviti besplatan račun na stranici Brave API-ja i definirati samu funkciju` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Unatoč tome što je LLM, jedna od ograničenja koje Llama 3.1 ima jest multimodalnost. To jest, mogućnost korištenja različitih vrsta ulaza poput slika kao upita i pružanja odgovora. Ova sposobnost je jedna od glavnih značajki Llama 3.2. Te značajke također uključuju: 

- Multimodalnost - mogućnost evaluacije kako tekstualnih, tako i slikovnih upita 
- Varijacije male do srednje veličine (11B i 90B) - pruža fleksibilne opcije za implementaciju, 
- Samo tekstualne varijacije (1B i 3B) - omogućuju modelu implementaciju na edge / mobilnim uređajima i pružaju nisku latenciju 

Podrška za multimodalnost predstavlja veliki korak u svijetu open source modela. Primjer koda ispod uzima i sliku i tekstualni upit kako bi dobio analizu slike od Llama 3.2 90B. 

### Multimodalna podrška s Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Učenje ovdje ne prestaje, nastavite putovanje

Nakon što završite ovu lekciju, pogledajte našu [kolekciju za učenje generativne AI](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) kako biste nastavili unapređivati svoje znanje o generativnoj AI!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
